In [1]:
import pandas as pd
from IPython.display import Audio, display
import soundfile as sf
from pydub import AudioSegment
import numpy as np
import os, gc , time, json
from io import BytesIO


AudioPATH= "./audio"
ModelId1="Qwen/Qwen3-ASR-1.7B"
ModelId2="Qwen/Qwen3-ASR-0.6B "

In [2]:
# !pip install pydub

In [3]:
## Convert refvoice  mp3 file to wave file

In [4]:
from pydub import AudioSegment


def convert_mp3_to_wav(src, dst):
    # Convert the MP3 file to a WAV file
    try:
        sound = AudioSegment.from_mp3(src)
        sound.export(dst, format="wav")
        print(f"Successfully converted {src} to {dst}")
    except Exception as e:
        print(f"An error occurred: {e}")

## Load Cantonese dataset, download from huggingface : a cleaned verison from AlienKevin/mixed_cantonese_and_english_speech:
https://huggingface.co/datasets/AlienKevin/mixed_cantonese_and_english_speech

In [5]:
df = pd.read_parquet("train-00000-of-00001.parquet",  engine='pyarrow')

In [6]:
df.columns

Index(['audio', 'sentence', 'topic'], dtype='object')

In [7]:
df

,audio,sentence,topic
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！,天氣
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。,天氣
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。,天氣
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候...,天氣
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Forecast話晚上嘅weather會變得好cold，會降到freezing point，...,天氣
...,...,...,...
14012,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,雖然香港天氣好hot，但我哋可以想下use less air con，改用風扇。,环境
14013,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Buy local 嘅商品都可以減少carbon footprint，唔再郵寄長途嘅貨品。,环境
14014,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,用solar power同wind power都唔錯，我哋要追求sustainable嘅生活方式。,环境
14015,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,"Reduce, Reuse, Recycle 呢個三R原則喺日常生活中實踐，即係好好嘅典範。",环境


In [8]:
df["audio"][0]

{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x00\x00\x00\x0f\x00\x00\x03Lavf58.76.100\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf3X\xc0\x00\x00\x00\x00\x00\x00\x00\x00\x00Info\x00\x00\x00\x0f\x00\x00\x00\xe4\x00\x00`\xe4\x00\x05\x07\t\r\x0f\x11\x13\x17\x19\x1b\x1e!#%(+-0257:<?BDFILNPSVX[^`behjlprtvz|~\x82\x84\x86\x88\x8c\x8e\x90\x92\x96\x98\x9a\x9d\xa0\xa2\xa4\xa7\xaa\xac\xaf\xb1\xb4\xb6\xb9\xbc\xbe\xc1\xc3\xc6\xc8\xcb\xcd\xd0\xd2\xd5\xd7\xda\xdd\xdf\xe1\xe4\xe7\xe9\xeb\xef\xf1\xf3\xf5\xf9\xfb\xfd\x00\x00\x00\x00Lavc58.13\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00$\x04\x15\x00\x00\x00\x00\x00\x00`\xe4\xc0zi\x0b\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf38\xc4\x00\x00\x00\x03H\x00\x00\x00\x00LAME3.100UUUUUUUUUUUUUUUUUUUUULAME3.100UUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUU\xff\xf38\xc4_\x00\x00\x03H\x00\x00\x00\x00UUUUUUUUUUUUUUUUUUUUUUUUUUUUUULAME3.100UUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUUU\xff\xf38\xc4\xa0\x00\x00\x03H\x00\x00\x00\x00U

In [9]:
df["topic"].unique()

array(['天氣', '食物', '旅遊', '娛樂', '體育', '本地時事', '購物', '學習', '工作', '健康同健身',
       '宠物', '科技同新闻', '电影同电视剧', '音乐同艺术', '爱好同兴趣', '历史同文学', '社交媒体同网络文化',
       '环境'], dtype=object)

## Checking the Audio and Text label

In [10]:
def playAudio(raw):
    display(Audio(raw))

In [11]:
index = 3

playAudio(df["audio"][index]["bytes"]) 
print(f"Index : {index}")
print(f"Text : {df["sentence"][index]}")
print(f"File: {df["audio"][index]["path"]}")

Index : 3
Text : 周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候要小心啲！
File: 1_4.mp3


## Prepare Audio File for Hong Kong Cantonese (Fine tune) Training Dataset

In [12]:
def generateAudioFile(raw : bytes, fileName: str, format_hint="mp3"):
    # Wrap bytes in memory buffer
    audio_buffer = BytesIO(raw)
    # Try to load (pydub guesses format from content or you can force it)
    # audio = AudioSegment.from_file(audio_buffer, format=format_hint)
    data, samplerate = sf.read(audio_buffer)
    # print(samplerate)
    # Write to new file (format guessed from extension)
    sf.write(fileName, data, samplerate)
    
    print(f"Saved via soundfile → {fileName}")
    
    

In [13]:
generateAudioFile(df["audio"][index]["bytes"], f"{AudioPATH}/{index}.wav")

Saved via soundfile → ./audio/3.wav


In [14]:
numberSample= 2000 #500
for idx in range(numberSample):
    generateAudioFile(df["audio"][idx]["bytes"], f"{AudioPATH}/{idx}.wav")

Saved via soundfile → ./audio/0.wav
Saved via soundfile → ./audio/1.wav
Saved via soundfile → ./audio/2.wav
Saved via soundfile → ./audio/3.wav
Saved via soundfile → ./audio/4.wav
Saved via soundfile → ./audio/5.wav
Saved via soundfile → ./audio/6.wav
Saved via soundfile → ./audio/7.wav
Saved via soundfile → ./audio/8.wav
Saved via soundfile → ./audio/9.wav
Saved via soundfile → ./audio/10.wav
Saved via soundfile → ./audio/11.wav
Saved via soundfile → ./audio/12.wav
Saved via soundfile → ./audio/13.wav
Saved via soundfile → ./audio/14.wav
Saved via soundfile → ./audio/15.wav
Saved via soundfile → ./audio/16.wav
Saved via soundfile → ./audio/17.wav
Saved via soundfile → ./audio/18.wav
Saved via soundfile → ./audio/19.wav
Saved via soundfile → ./audio/20.wav
Saved via soundfile → ./audio/21.wav
Saved via soundfile → ./audio/22.wav
Saved via soundfile → ./audio/23.wav
Saved via soundfile → ./audio/24.wav
Saved via soundfile → ./audio/25.wav
Saved via soundfile → ./audio/26.wav
Saved via s

In [15]:
import random

# Generate a random integer between 1 and 10 (inclusive)
random_integer = random.randint(1, numberSample)
print(random_integer)

1172


In [16]:
# generateAudioFile(df["audio"][random_integer]["bytes"], f"{AudioPATH}/ref_voice/alienkevin.wav")

### Resample ALL Audio file Qwen3 ASR requirement only support 24Khz

In [19]:
# import librosa
# import soundfile as sf

# # wav, sr = librosa.load(f"{AudioPATH}/2.wav", sr=None)
# # wav, sr = librosa.load(RefVoicePATHWav, sr=None)
# wav , sr = librosa.load(f"{AudioPATH}/ref_voice/alienkevin.wav", sr=None)
# print(f"Original Sample rate: {sr}")

# if sr != 24000:
#     newWav = librosa.resample(wav, orig_sr=sr, target_sr=24000)
#     # Add normalization to prevent clipping
#     newWav = librosa.util.normalize(newWav)
#     sf.write(f"{AudioPATH}/ref_voice/alienkevin.wav", newWav, 24000)
#     # print(f"New Sample rate: {tempSR}")
#     sr = 24000
# else:
#      # Normalize even if the sample rate is correct
#     newWav = librosa.util.normalize(wav)
#     sf.write(audioPath, newWav, targetSR)
#     print(f"Audio File ready support target sample rate: {targetSR}")

In [20]:
def batchResample(audioPath, targetSR=24000):
    wav, sr = librosa.load(audioPath, sr=None)
    if sr != 24000:
        newWav = librosa.resample(wav, orig_sr=sr, target_sr=24000)
        # Add normalization to prevent clipping
        newWav = librosa.util.normalize(newWav)
        sf.write(audioPath, newWav, targetSR)
        print(f"Saved resample soundfile → {audioPath}")
    else:
        # Normalize even if the sample rate is correct
        newWav = librosa.util.normalize(wav)
        sf.write(audioPath, newWav, targetSR)
        print(f"Audio File ready support target sample rate: {targetSR}")
    

In [20]:
# numberSample= 500
for idx in range(numberSample):
    batchResample(f"{AudioPATH}/{idx}.wav", targetSR=24000)

Saved resample soundfile → ./audio/0.wav
Saved resample soundfile → ./audio/1.wav
Saved resample soundfile → ./audio/2.wav
Saved resample soundfile → ./audio/3.wav
Saved resample soundfile → ./audio/4.wav
Saved resample soundfile → ./audio/5.wav
Saved resample soundfile → ./audio/6.wav
Saved resample soundfile → ./audio/7.wav
Saved resample soundfile → ./audio/8.wav
Saved resample soundfile → ./audio/9.wav
Saved resample soundfile → ./audio/10.wav
Saved resample soundfile → ./audio/11.wav
Saved resample soundfile → ./audio/12.wav
Saved resample soundfile → ./audio/13.wav
Saved resample soundfile → ./audio/14.wav
Saved resample soundfile → ./audio/15.wav
Saved resample soundfile → ./audio/16.wav
Saved resample soundfile → ./audio/17.wav
Saved resample soundfile → ./audio/18.wav
Saved resample soundfile → ./audio/19.wav
Saved resample soundfile → ./audio/20.wav
Saved resample soundfile → ./audio/21.wav
Saved resample soundfile → ./audio/22.wav
Saved resample soundfile → ./audio/23.wav
Sa

In [21]:
# extract Dataframe

In [22]:
selectedDF =df[:numberSample].copy()

In [23]:
selectedDF

,audio,sentence,topic
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！,天氣
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。,天氣
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。,天氣
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候...,天氣
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Forecast話晚上嘅weather會變得好cold，會降到freezing point，...,天氣
...,...,...,...
9995,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Social media post可以喺seconds內reach到全世界，但係影響卻可以l...,社交媒体同网络文化
9996,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,大部分嘅celebrities都會喺Twitter發放自己嘅最新消息，所以你想keep up...,社交媒体同网络文化
9997,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Snapchat嘅feature係照片分享後會即刻消失，好多teenagers都好鍾意呢種e...,社交媒体同网络文化
9998,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,TikTok上嘅short videos真係有creative 到爆，有時真係覺得say a...,社交媒体同网络文化


### Generate JSONL file for prepare Training data

In [23]:
def generateJSONFile(dataFrame, audioDirectory, refAudioPath, outputFile="train.jsonl"):
    with open(outputFile, "w", encoding="utf-8") as f:
        for i, row in dataFrame.iterrows():
            obj = {
                "audio": f"{audioDirectory}/{i}.wav",
                "text": row["sentence"],   # map sentence -> text
            }
            # ensure_ascii=False keeps Chinese characters, etc.
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [24]:
# # generateJSONFile(selectedDF, AudioPATH, RefVoicePATHWav, outputFile="train_raw.jsonl")
# generateJSONFile(selectedDF, AudioPATH, f"{AudioPATH}/ref_voice/alienkevin.wav", 
#                  outputFile="train_raw.jsonl")

In [25]:
# trainDF = pd.read_json("train_raw.jsonl", lines=True)

# print(trainDF.head())
# print(trainDF.columns)  # Should show: ['audio', 'text', 'ref_audio']

In [26]:
# trainDF

In [27]:
# trainDF["ref_audio"][0]

### # Download model into local Directory

In [28]:
# Using huggingface-cli (install if needed: pip install -U huggingface_hub[cli])

# !huggingface-cli download Qwen/Qwen3-ASR-1.7B --local-dir ./Qwen3-ASR-1.7B
!huggingface-cli download Qwen/Qwen3-ASR-0.6B --local-dir ./Qwen3-ASR-0.6B

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files:   0%|                                 | 0/10 [00:00<?, ?it/s]Downloading 'model.safetensors' to 'Qwen3-ASR-0.6B/.cache/huggingface/download/xGOKKLRSlIhH692hSVvI1-gpoa8=.79d6cbd4c98c7bbffe9db2edac07f56cd6637d0d5944b27f6c2b8353840323ea.incomplete'

model.safetensors:   0%|                            | 0.00/1.88G [00:00<?, ?B/s]Downloading 'chat_template.json' to 'Qwen3-ASR-0.6B/.cache/huggingface/download/BqEbVjUtvDkrcHAfaoo906xJg7Y=.c44736493efd71ec96218cc626904698cdb13235.incomplete'


preprocessor_config.json: 100%|████████████████| 330/330 [00:00<00:00, 1.32MB/s]


Download complete. Moving file to Qwen3-ASR-0.6B/preprocessor_config.json
.gitattributes: 0.00B [00:00, ?B/s]


chat_template.json: 0.00B [00:00, ?B/s]



generation_config.json: 100%|███████████████████| 142/142 [00:00<00:00, 615kB/s]
chat_template.json: 1.16kB [00:00, 470kB/s]
Download complete. Moving file to Qwen3-ASR-0

# Run Prepare data script from Qwen3 fine turning python script)

In [31]:
# %%time 
# ## Support generate both train and evaluate 
# !python prepare_train_evaluate_data.py \
# --device cuda:0 \
# --input_jsonl train_raw.jsonl \
# --output_train_jsonl train_prepared.jsonl \
# --output_eval_jsonl eval_prepared.jsonl \
# --eval_ratio 0.1 \
# --speaker_name hk_cantonese_speaker

Loading raw data from: train_raw.jsonl
Total samples: 10000
Train samples: 9000 | Eval samples: 1000
Encoding audio_codes for TRAINING dataset (this may take a while)...
✅ Training dataset saved: train_prepared.jsonl (9000 samples)
✅ Evaluation dataset saved: eval_prepared.jsonl (1000 samples)

🎉 All datasets prepared successfully!
   → Use --train_jsonl train_prepared.jsonl
   → Use --val_jsonl or --test_jsonl eval_prepared.jsonl
CPU times: user 3.28 s, sys: 584 ms, total: 3.86 s
Wall time: 3min 14s


# Run SFT using the prepared JSONL:
#### select batch size depend on your gpu memory: minimize at least GPU 12GB VRAM, otherwise error Out-of-Memory (OOM)
#### batch _size =1 or 2 for 12GB (3080 GPU),  batch_size =4 for 16GB GPU VRAM

### MLflow Trace support version

In [30]:
%%time
# !python sft_12hz_mlflow.py \
#   --init_model_path ./Qwen3-TTS-12Hz-0.6B-Base \
#   --output_model_path output \
#   --train_jsonl train_prepared.jsonl \
#   --batch_size 2 \
#   --lr 1e-5 \
#   --num_epochs 20 \
#   --gradient_accumulation_steps 8 \
#   --speaker_name speaker_test \
#   --mlflow_tracking_uri http://localhost:5000 


CPU times: user 11 μs, sys: 2 μs, total: 13 μs
Wall time: 25.5 μs


In [31]:
# upload final model to mlflow
import mlflow
# mlflow.log_artifacts("output/checkpoint-epoch-3", artifact_path="final_checkpoint")


ModuleNotFoundError: No module named 'mlflow'

## Test fine tuning